In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql import functions as F
from pyspark.sql.types import NumericType

In [2]:
# Inicializamos la Sesión de Spark

spark = SparkSession.builder \
    .appName("Analisis_Clientes_Behavioural") \
    .getOrCreate()


behavioural_df = spark.read.parquet("/home/jovyan/work/data/BEHAVIOURAL", header=True, inferSchema=True)
clientes_df = spark.read.parquet("/home/jovyan/work/data/CLIENTS", header=True, inferSchema=True)

#clientes_psdf = clientes_df.pandas_api()
#behavioural_psdf = behavioural_df.pandas_api()

# ANALISIS BEHAVIOURAL

In [3]:
## ANALISIS BEHAVIOURAL - Exploración Inicial

# Visualizamos las primeras filas del DataFrame BEHAVIOURAL
print("Primeras 5 filas de BEHAVIOURAL:")
behavioural_df.show(5, truncate=False)

# Visualizamos el esquema del DataFrame BEHAVIOURAL
print("Esquema de BEHAVIOURAL:")
behavioural_df.printSchema()

Primeras 5 filas de BEHAVIOURAL:
+------------------+------------+----------+--------------------+-----------------+------------------------+--------------------+------------------------+--------------------------+-------------------+-------------------+---------------+------------------+--------+
|CONTRACT_ID       |CLIENT_ID   |DATE      |CREDICT_CARD_BALANCE|CREDIT_CARD_LIMIT|CREDIT_CARD_DRAWINGS_ATM|CREDIT_CARD_DRAWINGS|CREDIT_CARD_DRAWINGS_POS|CREDIT_CARD_DRAWINGS_OTHER|CREDIT_CARD_PAYMENT|NUMBER_DRAWINGS_ATM|NUMBER_DRAWINGS|NUMBER_INSTALMENTS|CURRENCY|
+------------------+------------+----------+--------------------+-----------------+------------------------+--------------------+------------------------+--------------------------+-------------------+-------------------+---------------+------------------+--------+
|ES1821016961u00XXX|ES182147947X|2020-08-22|0.0                 |2700.0           |0.0                     |0.0                 |0.0                     |0.0            

In [4]:
## Estadísticas Descriptivas 

# Contar filas
n_filas_beh = behavioural_df.count()

print(f"Shape: ({n_filas_beh}, {len(behavioural_df.columns)})") # filas y columnas
print("\nEstadísticas Descriptivas:")

# Visitalizamos las estadísticas descriptivas usando describe()
behavioural_df.describe().show(truncate=False)

Shape: (1724854, 14)

Estadísticas Descriptivas:
+-------+------------------+------------+--------------------+------------------+------------------------+--------------------+------------------------+--------------------------+-------------------+-------------------+------------------+------------------+--------+
|summary|CONTRACT_ID       |CLIENT_ID   |CREDICT_CARD_BALANCE|CREDIT_CARD_LIMIT |CREDIT_CARD_DRAWINGS_ATM|CREDIT_CARD_DRAWINGS|CREDIT_CARD_DRAWINGS_POS|CREDIT_CARD_DRAWINGS_OTHER|CREDIT_CARD_PAYMENT|NUMBER_DRAWINGS_ATM|NUMBER_DRAWINGS   |NUMBER_INSTALMENTS|CURRENCY|
+-------+------------------+------------+--------------------+------------------+------------------------+--------------------+------------------------+--------------------------+-------------------+-------------------+------------------+------------------+--------+
|count  |1724854           |1724854     |1724854             |1724854           |1724854                 |1724854             |1724854                

### Analisis de valores nulos

In [5]:
## Recuento de Valores Nulos 

# Creamos una lista de expresiones de agregación(Si la columna es nula, cuenta 1, si no, cuenta 0)

nulos_expr = [
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in behavioural_df.columns
]

# Ejecuta la agregación una sola vez en el cluster
missing_values_beh = behavioural_df.agg(*nulos_expr).collect()[0]

print("Valores nulos por columna en BEHAVIOURAL:")
found_missing = False
for column in behavioural_df.columns:
    missing_count = missing_values_beh[column]
    if missing_count > 0:
        porcentaje = (missing_count / n_filas_beh) * 100
        print(f"  {column}: {missing_count} nulos ({porcentaje:.2f}%)")
        found_missing = True

if not found_missing:
    print("No hay valores nulos en el dataset BEHAVIOURAL.")

Valores nulos por columna en BEHAVIOURAL:
No hay valores nulos en el dataset BEHAVIOURAL.


### Analisis de duplicados

In [6]:
# Contar cuántas veces aparece cada fila
conteos = behavioural_df.groupBy(behavioural_df.columns).agg(F.count("*").alias("count"))

# Filas que están duplicadas (count > 1)
duplicados_totales = conteos.withColumn("duplicated_count", F.col("count") - 1)

# Suma total de duplicados, equivalente a pandas .duplicated().sum()
num_duplicados = duplicados_totales.agg(F.sum("duplicated_count")).collect()[0][0]

# Si no hay duplicados, num_duplicados será None → lo convertimos a 0
num_duplicados = num_duplicados if num_duplicados is not None else 0

print(f"Filas duplicadas en BEHAVIOURAL: {num_duplicados}")

Filas duplicadas en BEHAVIOURAL: 0


### Analisis de outliers

In [7]:
beh_numeric_cols = [f.name for f in behavioural_df.schema.fields 
                    if isinstance(f.dataType, NumericType)]

total_rows = behavioural_df.count()

In [8]:
umbral = 3.5
scale_factor = 1.482602218505602  # scale='normal'

for colname in beh_numeric_cols:
    # Filtramos filas con valor distinto de 0
    df_col = behavioural_df.filter(F.col(colname) != 0)

    total_col_rows = df_col.count()
    if total_col_rows == 0:
        print(f"\n⚠️ Columna {colname}: todas las filas son 0.")
        continue

    mediana = df_col.select(
        F.expr(f'percentile_approx({colname}, 0.5)').alias('median')
    ).collect()[0]['median']

    mad_raw = df_col.select(
        F.expr(f'percentile_approx(ABS({colname} - {mediana}), 0.5)').alias('mad')
    ).collect()[0]['mad']

    if mad_raw is None or mad_raw == 0:
        print(f"\ Columna {colname}: MAD = {mad_raw} incluso sin ceros.")
        continue

    mad_scaled = mad_raw * scale_factor

    robust_z = F.abs((F.col(colname) - mediana) / mad_scaled)

    count_outliers = df_col.filter(robust_z > umbral).count()
    perc_outliers = count_outliers / total_col_rows * 100

    print(f"\n📌 Columna: {colname}")
    print(f"   - Outliers (MAD): {count_outliers} ({perc_outliers:.2f}%)")
    print(f"   - Mediana = {mediana}, MAD_crudo = {mad_raw}, MAD_escalado = {mad_scaled}")



📌 Columna: CREDICT_CARD_BALANCE
   - Outliers (MAD): 37682 (4.88%)
   - Mediana = 1264.35, MAD_crudo = 722.3399999999999, MAD_escalado = 1070.9428865153363

📌 Columna: CREDIT_CARD_LIMIT
   - Outliers (MAD): 181508 (13.09%)
   - Mediana = 1620.0, MAD_crudo = 540.0, MAD_escalado = 800.6051979930251

📌 Columna: CREDIT_CARD_DRAWINGS_ATM
   - Outliers (MAD): 25414 (13.02%)
   - Mediana = 216.0, MAD_crudo = 172.8, MAD_escalado = 256.19366335776806

📌 Columna: CREDIT_CARD_DRAWINGS
   - Outliers (MAD): 36652 (13.14%)
   - Mediana = 227.07, MAD_crudo = 189.26999999999998, MAD_escalado = 280.61212189655527

📌 Columna: CREDIT_CARD_DRAWINGS_POS
   - Outliers (MAD): 14098 (12.10%)
   - Mediana = 171.52, MAD_crudo = 140.20000000000002, MAD_escalado = 207.86083103448544

📌 Columna: CREDIT_CARD_DRAWINGS_OTHER
   - Outliers (MAD): 753 (13.00%)
   - Mediana = 336.96, MAD_crudo = 283.5, MAD_escalado = 420.31772894633815

📌 Columna: CREDIT_CARD_PAYMENT
   - Outliers (MAD): 82903 (6.82%)
   - Mediana = 54

In [9]:
for colname in beh_numeric_cols:
    
    # Q1 y Q3 usando percentile_approx
    q1 = behavioural_df.select(F.expr(f"percentile_approx({colname}, 0.25)")).collect()[0][0]
    q3 = behavioural_df.select(F.expr(f"percentile_approx({colname}, 0.75)")).collect()[0][0]
    iqr = q3 - q1

    # Límites
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    # Filtrar outliers
    outliers_df = behavioural_df.filter(
        (F.col(colname) < lower) | (F.col(colname) > upper)
    )

    # Contar
    count_outliers = outliers_df.count()
    percentage_outliers = (count_outliers / n_filas_beh) * 100 if n_filas_beh > 0 else 0

    # Obtener lista de valores (limitado para que no explote)
    valores = (
        outliers_df.select(colname)
        .limit(50)  # QUITAR si quieres todos, pero puede ser gigante
        .toPandas()[colname]
        .values
    )

    print(f"\n📌 Columna: {colname}")
    print(f"   - Nº de outliers: {count_outliers}")
    print(f"   - Porcentaje: {percentage_outliers:.2f}%")
    print(f"   - Rango permitido: [{lower}, {upper}]")
    print(f"   - Ejemplos de valores outliers: {valores}")


📌 Columna: CREDICT_CARD_BALANCE
   - Nº de outliers: 108188
   - Porcentaje: 6.27%
   - Rango permitido: [-1634.715, 2724.5249999999996]
   - Ejemplos de valores outliers: [ 3340.26  2791.13  5671.22  2740.42  5220.46  8919.77  6301.09  4379.85
  4850.16  5168.47  3479.78  5379.17  3220.35  7104.55  5327.66  3404.21
  2865.9   3434.92  3612.75  2766.15  3154.6   8782.05  5565.45  3168.87
  5173.19  3187.24  2923.6   5140.11  2740.05  3418.24  2895.48  3717.17
  5150.54  5537.9   4058.25  5540.92  3191.98  2882.08  5105.95  3996.47
  3188.72  3330.09  5510.98 10480.9   4863.81  3535.49  2839.09  3235.85
  4175.68  5429.78]

📌 Columna: CREDIT_CARD_LIMIT
   - Nº de outliers: 178112
   - Porcentaje: 10.33%
   - Rango permitido: [-1890.0, 4590.0]
   - Ejemplos de valores outliers: [10800.  5400.  5400.  5400.  4860.  5400.  5400.  5400.  5400. 10800.
  5130.  5400.  5400.  5400.  5400.  5400.  5400. 10800.  9180.  5400.
  5400.  9180.  5400.  9180.  5400.  5400.  5670.  5400.  5130.  5400.

### Eliminación de la columna 'Currency'

In [10]:
behavioural_df = behavioural_df.drop("Currency")

print("Columna 'Currency' eliminada correctamente.")
print(f"Columnas finales: {len(behavioural_df.columns)}")

Columna 'Currency' eliminada correctamente.
Columnas finales: 13


### CONCLUSIONES DEL ANALISIS DE BEHAVIOURAL

Hemos podido observar que el dataset no presenta ni 'Missing Values' ni 'Valores nulos' ni duplicados, por lo que no hemps tenido que limpiar columnas o filas debido a estos factores.

Este tipo de datasets tienen valores muy bajos y otros muy altos, teniendo ina diatribución bastante segada ('long-tail'), por ello en el cáculo de outliers de los dos modos podemos observar que en algunas categorías (de carácter financial) se observa un cambio en el porcentaje del número de estos. De todos modos, no hemos considerado no eliminarlos ni hacer nada con ellos ya que tampoco presentan un porcentaje muy alto como para que nos afecte en la conclusión de los resultados y que se tenga que eliminar o sustituir por la media de resto de valores.

Por otro lado, hemos observado que la columna 'CURRENCY' es la moneda de movimiento y en todos los 'CLIENT_ID' es euro, por lo que eliminamos esa columna puesto es algo que se tiene en cuenta sin necesitar una columna.

In [ ]:
behavioural_df

# ANALISIS CLIENTS

In [11]:
## ANALISIS CLIENTS - Exploración Inicial

# Visualizamos las primeras filas del DataFrame CLIENTS
print("Primeras 5 filas de CLIENTS:")
clientes_df.show(5, truncate=False)

# Visualizamos el esquema del DataFrame CLIENTS
print("Esquema de CLIENTS:")
clientes_df.printSchema()

Primeras 5 filas de CLIENTS:
+------------+----------------------+-----------------+------+------------+--------------+-----------+---------+--------------+--------------+------------+------------------+-------------+--------------+-----------+-----------------+-------+-----------+------------------+------------------+------------------+---------------------+------------------+----------+--------------+----------+--------------------------+--------+---------------------+------------------------+------------------------+------------------------+---------------------------+---------------------------+---------------------------+-----------------------+-----------------------+-----------------------+----------------------+----------------------+-------------------+---------------------+-----------------+-------------------+----------------+
|CLIENT_ID   |NON_COMPLIANT_CONTRACT|NAME_PRODUCT_TYPE|GENDER|TOTAL_INCOME|AMOUNT_PRODUCT|INSTALLMENT|EDUCATION|MARITAL_STATUS|HOME_SITUATION|REGION_S

In [12]:
## Estadísticas Descriptivas de CLIENTS

# Contar filas
n_filas_cli = clientes_df.count()

print(f"Shape: ({n_filas_cli}, {len(clientes_df.columns)})")
print("\nEstadísticas Descriptivas:")

# Visitalizamos las estadísticas descriptivas usando describe()
clientes_df.describe().show(truncate=False)

Shape: (162977, 45)

Estadísticas Descriptivas:
+-------+------------+----------------------+-----------------+------+------------------+------------------+------------------+---------------------+--------------+-----------------------+--------------------+------------------+------------------+------------------+------------------+-----------------+------------------+------------------+------------------+---------------------+-------------------+---------------------+------------------+---------------+-------------------+----------+--------------------------+--------+---------------------+------------------------+------------------------+------------------------+---------------------------+---------------------------+---------------------------+-----------------------+-----------------------+-----------------------+----------------------+----------------------+-------------------+---------------------+------------------+-------------------+------------------+
|summary|CLIENT_ID   |NON_

In [13]:
## Recuento de Valores Nulos en CLIENTS (Optimizado para PySpark)

# Crea una lista de expresiones de agregación para CLIENTS

nulos_expr_cli = [
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in clientes_df.columns
]

# Ejecuta la agregación una sola vez en el cluster
missing_values_cli = clientes_df.agg(*nulos_expr_cli).collect()[0]

print("Valores nulos por columna en CLIENTS:")
found_missing_cli = False
for column in clientes_df.columns:
    missing_count = missing_values_cli[column]
    if missing_count > 0:
        porcentaje = (missing_count / n_filas_cli) * 100
        print(f"  {column}: {missing_count} nulos ({porcentaje:.2f}%)")
        found_missing_cli = True

if not found_missing_cli:
    print("No hay valores nulos en el dataset CLIENTS.")

Valores nulos por columna en CLIENTS:
  INSTALLMENT: 7 nulos (0.00%)
  EDUCATION: 39640 nulos (24.32%)
  MARITAL_STATUS: 2 nulos (0.00%)
  JOB_SENIORITY: 29174 nulos (17.90%)
  CAR_AGE: 107550 nulos (65.99%)
  FAMILY_SIZE: 2 nulos (0.00%)
  REACTIVE_SCORING: 91901 nulos (56.39%)
  PROACTIVE_SCORING: 337 nulos (0.21%)
  BEHAVIORAL_SCORING: 32246 nulos (19.79%)
  DAYS_LAST_INFO_CHANGE: 1 nulos (0.00%)
  NUMBER_OF_PRODUCTS: 21903 nulos (13.44%)
  EMPLOYER_ORGANIZATION_TYPE: 29464 nulos (18.08%)
  NUM_PREVIOUS_LOAN_APP: 8770 nulos (5.38%)
  LOAN_ANNUITY_PAYMENT_MAX: 8770 nulos (5.38%)
  LOAN_ANNUITY_PAYMENT_MIN: 8770 nulos (5.38%)
  LOAN_ANNUITY_PAYMENT_SUM: 8770 nulos (5.38%)
  LOAN_APPLICATION_AMOUNT_MAX: 8770 nulos (5.38%)
  LOAN_APPLICATION_AMOUNT_MIN: 8770 nulos (5.38%)
  LOAN_APPLICATION_AMOUNT_SUM: 8770 nulos (5.38%)
  LOAN_CREDIT_GRANTED_MAX: 8770 nulos (5.38%)
  LOAN_CREDIT_GRANTED_MIN: 8770 nulos (5.38%)
  LOAN_CREDIT_GRANTED_SUM: 8770 nulos (5.38%)
  LOAN_VARIABLE_RATE_MAX: 8770

In [14]:
columnas_a_filtrar = [
    "NUM_PREVIOUS_LOAN_APP",
    "LOAN_ANNUITY_PAYMENT_MAX",
    "LOAN_ANNUITY_PAYMENT_MIN",
    "LOAN_ANNUITY_PAYMENT_SUM",
    "LOAN_APPLICATION_AMOUNT_MAX",
    "LOAN_APPLICATION_AMOUNT_MIN",
    "LOAN_APPLICATION_AMOUNT_SUM",
    "LOAN_CREDIT_GRANTED_MAX",
    "LOAN_CREDIT_GRANTED_MIN",
    "LOAN_CREDIT_GRANTED_SUM",
    "LOAN_VARIABLE_RATE_MAX",
    "LOAN_VARIABLE_RATE_MIN",
    "NUM_STATUS_ANNULLED",
    "NUM_STATUS_AUTHORIZED",
    "NUM_STATUS_DENIED",
    "NUM_STATUS_NOT_USED",
    "NUM_FLAG_INSURED"
]

# Eliminar filas con null/NaN en cualquiera de esas columnas
clientes_df = clientes_df.dropna(subset=columnas_a_filtrar)

print("Filas eliminadas correctamente.")
print(f"Número final de filas: {clientes_df.count()}")
print(f"Número final de columnas: {len(clientes_df.columns)}")


columnas_drop = ["CAR_AGE", "REACTIVE_SCORING", "CURRENCY"]

clientes_df = clientes_df.drop(*columnas_drop)

print("Columnas eliminadas correctamente.")
print(f"Columnas finales: {len(clientes_df.columns)}")

Filas eliminadas correctamente.
Número final de filas: 154207
Número final de columnas: 45
Columnas eliminadas correctamente.
Columnas finales: 42


In [ ]:
# Contador de missing values (null o NaN) ----
miss_counts = clientes_df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in clientes_df.columns
]).collect()[0].asDict()

# Número total de filas para el porcentaje ----
total_filas = clientes_df.count()

# Mostrar resultados en tu formato ----
print("\n Missing Values por columna:")

for col, missing in miss_counts.items():
    porcentaje = (missing / total_filas) * 100
    print(f"{col}: {missing} missing ({porcentaje:.2f}%)")


 Missing Values por columna:
CLIENT_ID: 0 missing (0.00%)
NON_COMPLIANT_CONTRACT: 0 missing (0.00%)
NAME_PRODUCT_TYPE: 0 missing (0.00%)
GENDER: 0 missing (0.00%)
TOTAL_INCOME: 0 missing (0.00%)
AMOUNT_PRODUCT: 0 missing (0.00%)
INSTALLMENT: 7 missing (0.00%)
EDUCATION: 36058 missing (23.38%)
MARITAL_STATUS: 0 missing (0.00%)
HOME_SITUATION: 0 missing (0.00%)
REGION_SCORE: 0 missing (0.00%)
AGE_IN_YEARS: 0 missing (0.00%)
JOB_SENIORITY: 27700 missing (17.96%)
HOME_SENIORITY: 0 missing (0.00%)
LAST_UPDATE: 0 missing (0.00%)
OWN_INSURANCE_CAR: 0 missing (0.00%)
FAMILY_SIZE: 0 missing (0.00%)
PROACTIVE_SCORING: 295 missing (0.19%)
BEHAVIORAL_SCORING: 30198 missing (19.58%)
DAYS_LAST_INFO_CHANGE: 0 missing (0.00%)
NUMBER_OF_PRODUCTS: 20723 missing (13.44%)
OCCUPATION: 0 missing (0.00%)
DIGITAL_CLIENT: 0 missing (0.00%)
HOME_OWNER: 0 missing (0.00%)
EMPLOYER_ORGANIZATION_TYPE: 27969 missing (18.14%)
NUM_PREVIOUS_LOAN_APP: 0 missing (0.00%)
LOAN_ANNUITY_PAYMENT_MAX: 0 missing (0.00%)
LOAN_A

In [16]:
# Contar ceros por columna
ceros_dict = (
    clientes_df.select([
        F.count(F.when(F.col(c) == 0, c)).alias(c)
        for c in clientes_df.columns
    ])
    .collect()[0]
    .asDict()
)

# Mostrar como lista
print("📌 Valores 0 por columna:\n")
for col, zeros in ceros_dict.items():
    print(f"{col}: {zeros} ceros")

📌 Valores 0 por columna:

CLIENT_ID: 0 ceros
NON_COMPLIANT_CONTRACT: 141494 ceros
NAME_PRODUCT_TYPE: 0 ceros
GENDER: 0 ceros
TOTAL_INCOME: 0 ceros
AMOUNT_PRODUCT: 0 ceros
INSTALLMENT: 0 ceros
EDUCATION: 0 ceros
MARITAL_STATUS: 0 ceros
HOME_SITUATION: 0 ceros
REGION_SCORE: 0 ceros
AGE_IN_YEARS: 0 ceros
JOB_SENIORITY: 0 ceros
HOME_SENIORITY: 39 ceros
LAST_UPDATE: 7 ceros
OWN_INSURANCE_CAR: 0 ceros
FAMILY_SIZE: 0 ceros
PROACTIVE_SCORING: 0 ceros
BEHAVIORAL_SCORING: 0 ceros
DAYS_LAST_INFO_CHANGE: 17991 ceros
NUMBER_OF_PRODUCTS: 31879 ceros
OCCUPATION: 0 ceros
DIGITAL_CLIENT: 145322 ceros
HOME_OWNER: 0 ceros
EMPLOYER_ORGANIZATION_TYPE: 0 ceros
NUM_PREVIOUS_LOAN_APP: 0 ceros
LOAN_ANNUITY_PAYMENT_MAX: 200 ceros
LOAN_ANNUITY_PAYMENT_MIN: 67707 ceros
LOAN_ANNUITY_PAYMENT_SUM: 200 ceros
LOAN_APPLICATION_AMOUNT_MAX: 529 ceros
LOAN_APPLICATION_AMOUNT_MIN: 72400 ceros
LOAN_APPLICATION_AMOUNT_SUM: 529 ceros
LOAN_CREDIT_GRANTED_MAX: 112 ceros
LOAN_CREDIT_GRANTED_MIN: 60036 ceros
LOAN_CREDIT_GRANTED_S

In [17]:
# Contar total de filas
total_filas = clientes_df.count()

# Contar filas distintas
filas_unicas = clientes_df.dropDuplicates().count()

# Duplicadas = total - únicas
duplicadas = total_filas - filas_unicas

print(f"Filas duplicadas en CLIENTS: {duplicadas}")

Filas duplicadas en CLIENTS: 0


In [34]:
clientes_df

DataFrame[CLIENT_ID: string, NON_COMPLIANT_CONTRACT: int, NAME_PRODUCT_TYPE: string, GENDER: string, TOTAL_INCOME: double, AMOUNT_PRODUCT: double, INSTALLMENT: double, EDUCATION: string, MARITAL_STATUS: string, HOME_SITUATION: string, REGION_SCORE: double, AGE_IN_YEARS: double, JOB_SENIORITY: double, HOME_SENIORITY: double, LAST_UPDATE: double, OWN_INSURANCE_CAR: string, FAMILY_SIZE: double, PROACTIVE_SCORING: double, BEHAVIORAL_SCORING: double, DAYS_LAST_INFO_CHANGE: double, NUMBER_OF_PRODUCTS: double, OCCUPATION: string, DIGITAL_CLIENT: int, HOME_OWNER: string, EMPLOYER_ORGANIZATION_TYPE: string, NUM_PREVIOUS_LOAN_APP: double, LOAN_ANNUITY_PAYMENT_MAX: double, LOAN_ANNUITY_PAYMENT_MIN: double, LOAN_ANNUITY_PAYMENT_SUM: double, LOAN_APPLICATION_AMOUNT_MAX: double, LOAN_APPLICATION_AMOUNT_MIN: double, LOAN_APPLICATION_AMOUNT_SUM: double, LOAN_CREDIT_GRANTED_MAX: double, LOAN_CREDIT_GRANTED_MIN: double, LOAN_CREDIT_GRANTED_SUM: double, LOAN_VARIABLE_RATE_MAX: double, LOAN_VARIABLE_RATE_

In [18]:
behavioural_df.write.mode("overwrite").parquet("/home/jovyan/work/data/BEHAVIOURAL_CLEAN")
clientes_df.write.mode("overwrite").parquet("/home/jovyan/work/data/CLIENTS_CLEAN")

In [21]:
df_cli = spark.read.parquet("/home/jovyan/work/data/CLIENTS_CLEAN")

# Convertir a Pandas
df_clients_pd = df_cli.toPandas()

# Guardar CSV
df_clients_pd.to_csv("/home/jovyan/work/data/CLIENTS_CLEAN.csv", index=False)


In [ ]:
df_beh = spark.read.parquet("/home/jovyan/work/data/BEHAVIOURAL_CLEAN")

# Convertir a Pandas
df_beh_pd = df_beh.toPandas()

# Guardar CSV
df_beh_pd.to_csv("/home/jovyan/work/data/BEHAVIOURAL_CLEAN.csv", index=False)

In [36]:
df_beh

DataFrame[CONTRACT_ID: string, CLIENT_ID: string, DATE: date, CREDICT_CARD_BALANCE: double, CREDIT_CARD_LIMIT: double, CREDIT_CARD_DRAWINGS_ATM: double, CREDIT_CARD_DRAWINGS: double, CREDIT_CARD_DRAWINGS_POS: double, CREDIT_CARD_DRAWINGS_OTHER: double, CREDIT_CARD_PAYMENT: double, NUMBER_DRAWINGS_ATM: double, NUMBER_DRAWINGS: int, NUMBER_INSTALMENTS: double]